In [41]:
import os
print(os.listdir(r"/mnt/2tb-samsung/zychin/HCJC2026/dogncat/images")[:10])

['MSR-LA - 3467.docx', 'PetImages', 'readme[1].txt']


In [ ]:
print(os.listdir(r"/mnt/2tb-samsung/zychin/HCJC2026/dogncat/images/PetImages"))

['mnt', 'dogs_vs_cats_yolov8.pth', 'dogs_vs_cats_yolov8', '.conda', 'yolov8n-cls.pt', 'images', 'dogs_vs_cats.ipynb', 'archive.zip']


In [42]:
cat_images = os.listdir(r"/mnt/2tb-samsung/zychin/HCJC2026/dogncat/images/PetImages/Cat")
dog_images = os.listdir(r"/mnt/2tb-samsung/zychin/HCJC2026/dogncat/images/PetImages/Dog")

print(f"Cats: {len(cat_images)}")
print(f"Dogs: {len(dog_images)}")
print(f"First 5 cat files: {cat_images[:5]}")

Cats: 12501
Dogs: 12501
First 5 cat files: ['12354.jpg', '6468.jpg', '2656.jpg', '9920.jpg', '10392.jpg']


In [43]:
import os
import shutil
import random
from PIL import Image

# Define paths
base = r"/mnt/2tb-samsung/zychin/HCJC2026/dogncat/images/PetImages"
output = r"/mnt/2tb-samsung/zychin/HCJC2026/dogncat/dataset"

# Clean up existing split folder if it exists to prevent old bad images from remaining
if os.path.exists(output):
    shutil.rmtree(output)

# Split ratios
train_ratio = 0.70
val_ratio   = 0.15

random.seed(42)  # makes the split reproducible

for animal in ["Cat", "Dog"]:
    all_files = os.listdir(os.path.join(base, animal))
    
    # Filter out corrupted files before splitting
    valid_images = []
    for f in all_files:
        if f.endswith(".jpg"):
            file_path = os.path.join(base, animal, f)
            try:
                with Image.open(file_path) as img:
                    img.verify()  # Only keep it if PIL can safely read it
                valid_images.append(f)
            except Exception:
                print(f"Skipping corrupt source image: {file_path}")

    random.shuffle(valid_images)

    n = len(valid_images)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))

    splits = {
        "train": valid_images[:train_end],
        "val":   valid_images[train_end:val_end],
        "test":  valid_images[val_end:]
    }

    for split, files in splits.items():
        dest_folder = os.path.join(output, split, animal)
        os.makedirs(dest_folder, exist_ok=True)
        for f in files:
            src = os.path.join(base, animal, f)
            dst = os.path.join(dest_folder, f)
            shutil.copy(src, dst)
        print(f"{split}/{animal}: {len(files)} images copied")

Skipping corrupt source image: /mnt/2tb-samsung/zychin/HCJC2026/dogncat/images/PetImages/Cat/666.jpg
train/Cat: 8749 images copied
val/Cat: 1875 images copied
test/Cat: 1875 images copied


/home/zychin/miniforge3/envs/dogncat/lib/python3.10/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Skipping corrupt source image: /mnt/2tb-samsung/zychin/HCJC2026/dogncat/images/PetImages/Dog/11702.jpg
train/Dog: 8749 images copied
val/Dog: 1875 images copied
test/Dog: 1875 images copied


In [44]:
for split in ["train", "val", "test"]:
    for animal in ["Cat", "Dog"]:
        path = os.path.join(output, split, animal)
        print(f"{split}/{animal}: {len(os.listdir(path))} images")

train/Cat: 8749 images
train/Dog: 8749 images
val/Cat: 1875 images
val/Dog: 1875 images
test/Cat: 1875 images
test/Dog: 1875 images


In [45]:
#data prep (datasets and dataloaders)
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transformations
# Models expect images to be the same size, and normalized
transform = transforms.Compose([
    transforms.Resize((224, 224)),       # resize all images to 224x224
    transforms.ToTensor(),               # convert image to numbers (0-1)
    transforms.Normalize(                # standardize the numbers
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load datasets
train_data = datasets.ImageFolder(os.path.join(output, "train"), transform=transform)
val_data   = datasets.ImageFolder(os.path.join(output, "val"),   transform=transform)
test_data  = datasets.ImageFolder(os.path.join(output, "test"),  transform=transform)

# DataLoaders (feed images to model in batches)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False)

print(f"Classes: {train_data.classes}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Classes: ['Cat', 'Dog']
Train batches: 547
Val batches:   118
Test batches:  118


In [46]:
#building the model
import torch.nn as nn
from torchvision import models

# Load a pretrained ResNet18 (already trained on millions of images)
model = models.resnet18(pretrained=True)

# ResNet18 was built for 1000 classes — we only need 2 (cat/dog)
# So we replace the last layer
model.fc = nn.Linear(model.fc.in_features, 2)

# Move model to GPU since we have one
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Using device: {device}")
print(f"Model ready!")

/home/zychin/miniforge3/envs/dogncat/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zychin/miniforge3/envs/dogncat/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
76.7%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/zychin/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


Using device: cuda
Model ready!


In [47]:
#loss function and optimizer
import torch.optim as optim

criterion = nn.CrossEntropyLoss()  # for classification

#freeze all layers
for param in model.parameters():
    param.requires_grad = False

#repalce final layer
model.fc = nn.Linear(model.fc.in_features, 2)

#move to gpu/cpu
model = model.to(device)

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [48]:
# Unfreeze the whole model this time
for param in model.parameters():
    param.requires_grad = True

# Lower learning rate for pretrained layers, higher for the final layer
optimizer = optim.Adam([
    {"params": list(model.parameters())[:-2], "lr": 1e-5},  # pretrained layers
    {"params": model.fc.parameters(),         "lr": 1e-3},  # final layer
])

# Train for 5 more epochs
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.0816
Epoch 2, Loss: 0.0330
Epoch 3, Loss: 0.0153
Epoch 4, Loss: 0.0118
Epoch 5, Loss: 0.0050


In [49]:
#validation, checks if the model is learning, not memorising (overfitting/underfitting)
model.eval()  # evaluation mode
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100 * correct / total:.2f}%")

Validation Accuracy: 98.83%


In [50]:
torch.save(model.state_dict(), "dogs_vs_cats.pth")

In [51]:
import gradio as gr
import torch
import torch.nn.functional as F
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18()
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("dogs_vs_cats.pth", map_location=device))
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def predict(image):
    image = Image.fromarray(image).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        probs = F.softmax(outputs, dim=1)[0]

    return {"Cat": float(probs[0]), "Dog": float(probs[1])}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=2),
    title="Cat vs Dog Classifier",
    description="Upload a photo to find out!"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [53]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [57]:
accuracy_score(all_labels, all_preds)
classification_report(all_labels, all_preds)
confusion_matrix(all_labels, all_preds)

#Top Left, true cats predicted as cats
#Top Right, cats predicted as dogs 
#Bottom Left, dogs predicted as cats
#Bottom Right, true dogs predicted as dogs



array([[1857,   18],
       [  16, 1859]])

In [ ]:
# =====================================================
# RESNET18: TEST ACCURACY (same metric as YOLOv8)
# =====================================================
# class_names matches the folder order ImageFolder uses: ['Cat', 'Dog']
class_names = test_data.classes  # ['Cat', 'Dog']

model.eval()
resnet_correct = 0
resnet_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        resnet_correct += (predicted == labels).sum().item()
        resnet_total += labels.size(0)

resnet_accuracy = 100 * resnet_correct / resnet_total
print(f"ResNet18 Test Accuracy: {resnet_accuracy:.2f}%")

#computes the percentage of predictions that are correct, not f1score, 
# precision, recall, etc. which are all different metrics 
# that can be used to evaluate a model's performance.

ResNet18 Test Accuracy: 99.09%
